<div align="center">
<h1 style="color:rgb(121, 65, 6);">Speech Emotion Recognition</h1>
<h3>Feature Engineering</h3>
<a href="">Give Feedback</a> | <a href="">Bug report</a>

</div>

**Tags:** #ser #audio #feature-engineering #mel-scale

**Author:** [Ndeye Awa Salane](https://www.linkedin.com/in/ndeye-awa-salane-a93667230/)

**Last update:** 2025-04-18 (Created: 2025-04-16)

**Description:** This notebook explores audio features in general, with a special focus on those commonly used in machine learning models. We will extract a variety of audio features, visualize them whenever possible, and then transform these features into formats that are both usable and effective for classification tasks. The goal is to understand how different audio features capture important characteristics of sound and how they can be used to improve model performance.

## Table of Contents

1. [Import libraries](#import-libraries)
2. [Landscape of audio features](#landscape-of-audio-features)
3. [Feature Extraction](#feature-extraction)
4. [Feature Engineering and Preprocessing](#feature-engineering-and-preprocessing)
6. [Conclusions](#conclusions)
7. [Resources](#resources)

In [ ]:
# pip install resampy  (uncomment if needed)

# Import libraries

In [ ]:
import sys, importlib
sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))

import IPython.display as ipd
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Force-reload so updated features.py is picked up
import src.config, src.features, src.training
for mod in [src.config, src.features, src.training]:
    importlib.reload(mod)

from src.config import RAVDESS_DIR, CREMA_DIR, DATA_RAW, DATA_PROCESSED, N_MFCC, SR, DURATION
from src.features import extract_features, extract_features_for_dataset, get_feature_names
from src.training import save_processed

sns.set(style="whitegrid")

print(f"Feature extraction config: N_MFCC={N_MFCC}, SR={SR}, DURATION={DURATION}")
print(f"Features per sample: {len(get_feature_names(N_MFCC))}")

# Landscape of audio features

<div align="center">
    <img style='width:400px' src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*xTYCtcx_7otHVu-uToI9dA.png">
    <a style='text-decoration:underline; font-size:8px' href='https://aavos.eu/glossary/fourier-transform/'>Source: Aavos International</a>
</div>

Features are quantifiable properties of audio signals that allow us to describe and analyze sound in a structured way. As an audio is fundamentally a signal, it can be represented in the time domain as a waveform, where each moment in the audio corresponds to an amplitude value thus showing how the signal changes over time. Time-domain features, such as the amplitude envelope, root mean square (RMS) energy, and zero-crossing rate, capture the temporal characteristics of the audio directly from this waveform representation.

Alternatively, an audio can also be represented in the frequency domain, where the signal is considered in terms of its frequency components. In this view, we observe how much energy/amplitude is present at each frequency, rather than at each moment in time.  This transformation from time to frequency domain is typically done mathematical tools like the <a href='https://en.wikipedia.org/wiki/Fourier_transform'>Fourier Transform</a>.

Other features are the Mel Frequency Cepstral Coefficients (MFCCs) which are a way to capture the characteristics of audio by representing how humans naturally hear sound (Mel Scale) that basically reflects how our ears are more sensitive to differences in lower frequencies than higher ones. The result is a set of numbers (coefficients) that sum up the shape of the sound’s spectrum(Check this out for more about the Mel spectogram: [Understanding the Mel Spectrogram](https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53))

When dealing with music/song audio, we might wanna use features like chroma, which capture the intensity of each of the twelve different pitch classes (C, C#, D....) present in the audio, regardless of octave. 

Read more about audio features here: [Audio Features](https://ravinkumar.com/GenAiGuidebook/audio/audio_feature_extraction.html) (Very detailed overview by the way :) )


For our classification task, we will focus more on the Mel coefficients and see if potentially we can use (or cross them with) other relevant features in the time domain or frequency domain even though the original project only used Mel coefficients as features.

# Feature Extraction

Okay… let's take a look at how a Mel spectrogram is represented and how the Mel coefficients are represented as data structures depending on the emotions and see if we can see any correlations even before an actual model/classifier.

Just to see those relationships or correlations maybe, we will plot again the waveform of the audio and then the Mel spectrogram/coefficients.



In [ ]:
sample_path = str(RAVDESS_DIR / "Actor_23" / "03-01-07-01-01-01-23.wav")
audio_data, sr = librosa.load(sample_path)

print(f"Sampling Rate: {sr}")
print(f"Audio Duration: {librosa.get_duration(y=audio_data, sr=sr)} seconds")

plt.figure(figsize=(10, 4))
librosa.display.waveshow(audio_data, sr=sr)
plt.title("Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()

ipd.Audio(sample_path)

In [ ]:
# Generate a spectrogram
spectrogram = librosa.feature.melspectrogram(y=audio_data, sr=sr)
log_spectrogram = librosa.power_to_db(spectrogram)

# Plot the spectrogram
plt.figure(figsize=(12, 4))
librosa.display.specshow(log_spectrogram, sr=sr, x_axis='time', y_axis='mel', cmap='coolwarm')
plt.title("Mel Spectrogram")
plt.colorbar(format='%+2.0f dB')
plt.show()

This is a time-frequency representation of the audio: in the x-axis, we have the time in seconds in this case and in the y-axis, we have the frequency in Hertz. The decibels (dB)represents the intensity/amplitude of each frequency and as you might guess high decibels represented by redish areas means loud sounds and lower decibels are presented by the blue zones, represent the quieter/calmer sounds and/or maybe moments of silence in the audio.

In [ ]:
# Extract MFCCs from the audio file
mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=40)

# Plot MFCCs
plt.figure(figsize=(12, 4))
librosa.display.specshow(mfccs, sr=sr, x_axis='time', cmap='viridis')
plt.title("MFCCs")
plt.colorbar()
plt.show()

The plots are quite similar, just the representations differs as these are just representation of the audio in different domains using : time domain (waveform), time-frequency domain (Mel scale)...

In [ ]:
mfccs = pd.DataFrame(mfccs)
mfccs

In [ ]:
# take the mean of each row of mfccs and plot it
plt.figure(figsize=(12, 4))
plt.plot(mfccs.mean(axis=0))    
plt.title("Mean MFCCs")

# plt.xlabel("MFCC Coefficients")
plt.ylabel("Mean Value")
plt.grid()
plt.show()

In [ ]:
fem_path = str(RAVDESS_DIR / "Actor_23" / "03-01-07-01-01-01-23.wav")
male_path = str(CREMA_DIR / "1001_DFA_DIS_XX.wav")

fem, sr = librosa.load(fem_path, duration=2.5, sr=22050*2, offset=0.5)
fem_mfccs = librosa.feature.mfcc(y=fem, sr=sr, n_mfcc=40)

male, sr = librosa.load(male_path, duration=2.5, sr=22050*2, offset=0.5)
male_mfccs = librosa.feature.mfcc(y=male, sr=sr, n_mfcc=40)

plt.figure(figsize=(12, 4))
plt.plot(fem_mfccs.mean(axis=0), label="Female")
plt.plot(male_mfccs.mean(axis=0), label="Male")
plt.title("Mean MFCCs")
plt.ylabel("Mean Value")
plt.legend()
plt.grid()
plt.show()

Well nice disco

In [ ]:
from src.data_loader import load_all_datasets, save_raw_dataset

# Rebuild the raw dataset with absolute paths (fixes old Windows backslash paths)
dataset = load_all_datasets()
save_raw_dataset(dataset)
print(f"Raw dataset: {dataset.shape}")
print(f"Sample path: {dataset['path'].iloc[0]}")
dataset.head()

In [ ]:
# This takes ~10 mins — extracts MFCCs for every audio file using src/features.py
df_features = extract_features_for_dataset(dataset)
print(f"Shape: {df_features.shape}")
df_features.head()

In [ ]:
# Save the UN-SCALED feature dataset — scaling happens later in the modelling notebook
# to avoid data leakage (scaler must be fit on train set only)
save_processed(df_features, "dataset.csv")
print("Saved to data/processed/dataset.csv (unscaled)")

# Save features (unscaled)

> **Note:** We intentionally do NOT scale here. Scaling happens in the modelling notebook *after* the train/test split, so the scaler is fit on train data only — no data leakage.

In [ ]:
# Quick sanity check — the processed dataset should have labels + source + path + feature columns
df_check = pd.read_csv(DATA_PROCESSED / "dataset.csv")
print(f"Columns: {len(df_check.columns)}, Rows: {len(df_check)}")
df_check.head()

# Conclusions

# Resources

- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Fourier Transform](https://en.wikipedia.org/wiki/Fourier_transform)
- [Audio Features](https://ravinkumar.com/GenAiGuidebook/audio/audio_feature_extraction.html)
- [Understanding the Mel Spectrogram](https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53)